# 02 · What the LLM sees

In a paid run the proposer is a language model, and each request gives it two texts: a
**system prompt** (the instructions, the same for every task) and one user message, the
**belief packet** (the state of the run, built from public data only). Only when a reply fails
validation does the one repair turn add the failed reply and the error message. This notebook
shows both texts for one round of the run that notebook 01 made, then the paragraphs that are
appended to the system prompt when one request asks for several candidates or offers an
archive review. Every sha256 below is computed in the notebook.

- **Needs:** the run folder of [notebook 01](01_engine_basics.ipynb),
  `examples/runs/quickstart_synthcount/`. Run notebook 01 first.
- **Runtime:** a few seconds. No API call is made.

Terms are defined in the glossary of [notebook 00](00_how_it_works.ipynb); the paper's
request protocol is described in [`FLOW.md`](../FLOW.md#the-protocol-of-the-paper-runs) and, in
detail, in section 12 of notebook 00.

## 1. Setup

In [1]:
import hashlib
import inspect
import json
import logging
import os
import sys
import textwrap
from pathlib import Path


def find_repo_root(start):
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file() and (folder / "src" / "bayes_model_discovery").is_dir():
            return folder
    raise FileNotFoundError("start Jupyter inside the repository checkout")


REPO = find_repo_root(Path.cwd().resolve())
os.chdir(REPO)
try:
    import bayes_model_discovery  # noqa: F401
except ImportError:  # not installed: use the source tree directly
    sys.path.insert(0, str(REPO / "src"))
logging.getLogger("pytensor.tensor.blas").setLevel(logging.ERROR)


def relative(path):
    """A path relative to the repository root, for printing."""
    try:
        return Path(path).resolve().relative_to(REPO)
    except ValueError:
        return Path(path).name


RUN = Path("examples/runs/quickstart_synthcount")
if not (RUN / "summary.json").is_file():
    raise FileNotFoundError(
        f"{RUN} does not exist yet. Run notebook 01_engine_basics.ipynb first: "
        "it makes this run folder in about five minutes, with no API key.")
config = json.loads((RUN / "config.json").read_text())
print(f"run folder: {RUN}\nproposer:   {config['proposer']}\nprompt:     {config['prompt_version']}")

run folder: examples/runs/quickstart_synthcount
proposer:   mock_fixed_template_responses_NOT_LLM
prompt:     v2.0


The run used the mock proposer, so no request was actually sent. The mock path still builds
the packet, writes it to `packets/tNN.json`, and records which system prompt a real request
would have carried. That makes a mock run the cheapest way to look at what a language model
would receive.

## 2. The user message: one belief packet

`packets/t01.json` holds the packet of round 1 exactly as it is sent: the runner writes the
same text it hands to the proposer. `belief_packet.serialize_packet` produces it as compact
JSON on one line, with sorted keys and every number rounded to four significant digits, so
that a packet is a stable, hashable record. The proposer puts it unchanged into the only user
message: `messages = [{"role": "user", "content": packet_text}]` in
`count_proposer.AnthropicCountProposer.propose_one` (the OpenAI transports send the same
message after a system message).

In [2]:
from bayes_model_discovery.experimental import belief_packet

t = 1
packet_path = RUN / "packets" / f"t{t:02d}.json"
packet_text = packet_path.read_text()
WIDTH = 120
print(f"{packet_path}: one line of {len(packet_text.encode()):,} bytes, printed in slices of {WIDTH} characters\n")
for start in range(0, len(packet_text), WIDTH):
    print(packet_text[start:start + WIDTH])

examples/runs/quickstart_synthcount/packets/t01.json: one line of 3,805 bytes, printed in slices of 120 characters

{"archive":[{"expression":"b0+b1*z+b2*square(z)","family":"binomial","model_id":"2bf731df9a5e18b9dce7","outcome":"active
","proposed_round":0,"reason":{"active_rounds":[0,1],"admitted_round":0}},{"expression":"b0+b1*log(1+u)","family":"binom
ial","model_id":"725dc31cf0bbf6abb59f","outcome":"active","proposed_round":0,"reason":{"active_rounds":[0,1],"admitted_r
ound":0}},{"expression":"b0+b1*z","family":"binomial","model_id":"e2a71965fd9c8137e3b5","outcome":"active","proposed_rou
nd":0,"reason":{"active_rounds":[0,1],"admitted_round":0}},{"expression":"b0","family":"binomial","model_id":"fd2262c8fc
36fdd43a04","outcome":"active","proposed_round":0,"reason":{"active_rounds":[0,1],"admitted_round":0}}],"best_model":"b0
+b1*z | binomial","best_model_id":"e2a71965fd9c8137e3b5","best_model_residual_definitions":{"pred_mean":"posterior_predi
ctive_mean_of_the_outcome","pred_sd":

The request record of that round names the packet by its size and sha256; recomputing both
from the file gives the same values. Parsing the packet and serializing it again gives the same
bytes.

In [3]:
digest = belief_packet.packet_digest(packet_text)
transport = json.loads((RUN / "proposals" / f"t{t:02d}.json").read_text())["transport"]
print("computed from the file:   ", digest)
print("recorded with the request:", {key: transport[key] for key in ("packet_bytes", "packet_sha256")})
packet = json.loads(packet_text)
print("serialize_packet(json.loads(text)) == text:", belief_packet.serialize_packet(packet) == packet_text)

computed from the file:    {'packet_bytes': 3805, 'packet_sha256': '4091eb83a757111863c7f2509ba893126e9e6e421dd8e34ed27fadb7652e32e2'}
recorded with the request: {'packet_bytes': 3805, 'packet_sha256': '4091eb83a757111863c7f2509ba893126e9e6e421dd8e34ed27fadb7652e32e2'}
serialize_packet(json.loads(text)) == text: True


## 3. What is in the packet

- `interface`: the only task-specific part. Input names and ranges (`u` in [0, 1]), derived
  inputs (`z = 2*u-1`), the outcome type and support (0 to 50 successes out of 50 trials) and
  the link (logit). No task name, no story, no true parameter.
- `observations` and `n_public`: all public data so far, with the round of each observation.
- `pool`: every active model with its expression, family, evidence weight, log evidence,
  posterior mean and sd of each coefficient, and its history of one-step log scores.
- `best_model`, `best_model_id`, `best_model_residuals`, `best_model_residual_definitions`:
  standardized residuals of the highest-weight model on the public data.
- `archive`: every model proposed so far in the run, with its outcome (`active`,
  `retired_by_evidence`, `rejected`, `duplicate`, `fit_failed`) and a structured reason.
- `constraints`: what the run allows (below).
- `packet_version`, `round`.

In [4]:
for key, value in packet.items():
    size = f"{len(value)} items" if isinstance(value, (list, dict)) else json.dumps(value)
    print(f"{key:34} {size}")
print("\nconstraints:")
for key, value in packet["constraints"].items():
    print(f"  {key:32} {json.dumps(value)}")

archive                            4 items
best_model                         "b0+b1*z | binomial"
best_model_id                      "e2a71965fd9c8137e3b5"
best_model_residual_definitions    3 items
best_model_residuals               5 items
constraints                        9 items
interface                          6 items
n_public                           5
observations                       5 items
packet_version                     "v2"
pool                               4 items
round                              1

constraints:
  allowed_families                 ["binomial", "beta_binomial", "zi_binomial"]
  family_expansion                 true
  initial_family                   "binomial"
  initial_structures               ["b0", "b0+b1*z", "b0+b1*z+b2*square(z)", "b0+b1*log(1+u)"]
  max_distinct_mean_coefficients   6
  positive_mean_required_families  []
  reserved_names                   ["exp", "log", "pi", "power", "rho", "sqrt", "square", "tanh"]
  roles                

Nothing in the packet comes from the held-out evaluation set. That is enforced by the builder's
signature rather than by a check: `build_belief_packet` has no argument through which held-out
inputs, outcomes or scores could arrive.

In [5]:
print("build_belief_packet" + str(inspect.signature(belief_packet.build_belief_packet)))

build_belief_packet(public_observations, active_fits, archive, constraints, interface, round)


## 4. The system prompt

`config.json` records the version of the system prompt the run sends (`prompt_version`) and the
sha256 of its file (`prompt_sha256`). `count_proposer.prompt_version_file` maps a version to
its file in `src/bayes_model_discovery/experimental/prompts/`.

In [6]:
from bayes_model_discovery.experimental import count_proposer

version = config["prompt_version"]
prompt_path = count_proposer.prompt_version_file(version)
prompt_bytes = prompt_path.read_bytes()
prompt_sha256 = hashlib.sha256(prompt_bytes).hexdigest()
lines = prompt_bytes.decode().splitlines()
print(f"prompt_version {version}: {relative(prompt_path)}, {len(prompt_bytes):,} bytes, {len(lines)} lines")
print(f"sha256 computed here: {prompt_sha256}")
print(f"sha256 in config.json: {config['prompt_sha256']}")
print(f"equal: {prompt_sha256 == config['prompt_sha256']}\n\nfirst lines:\n")
for line in lines[:3]:
    print(textwrap.fill(line, 100))

prompt_version v2.0: src/bayes_model_discovery/experimental/prompts/proposer_system_v2_0.txt, 2,077 bytes, 11 lines
sha256 computed here: f095edf1adb3b6533a9e68e73f98ca1dee3f23cd077e424b092393cb32b93aef
sha256 in config.json: f095edf1adb3b6533a9e68e73f98ca1dee3f23cd077e424b092393cb32b93aef
equal: True

first lines:

Propose one new candidate model for the process described only by the data and the structured fields
in the user message. A Bayesian engine fits and scores every candidate.

The user message is a JSON belief packet: interface (outcome type and support, input names and
ranges, the fixed link), observations (all public data so far), pool (each active model with
expression, family, evidence weight, log evidence, one-step predictive log-score history, and
posterior mean and sd of each coefficient), best_model_residuals (standardized residuals of the
current best model on the public data, sorted by input), archive (every earlier proposal in this run
with its outcome and reason),

With one candidate per request (the default, and what this run uses), the system text is the
file, byte for byte. `composed_system_prompt` returns that text together with the identity
block that is written into every request record. So a request is these two texts:

In [7]:
system_text, identity = count_proposer.composed_system_prompt(1)
print("system text == prompt file:", system_text.encode() == prompt_bytes)
print("identity block:", identity, "\n")
request = {
    "system": f"<{relative(prompt_path)}, {len(system_text.encode()):,} bytes>",
    "messages": [{"role": "user", "content": f"<{packet_path}, {len(packet_text.encode()):,} bytes>"}],
}
print(json.dumps(request, indent=2))

system text == prompt file: True
identity block: {'prompt_version': 'v2.0', 'prompt_sha256': 'f095edf1adb3b6533a9e68e73f98ca1dee3f23cd077e424b092393cb32b93aef'} 

{
  "system": "<src/bayes_model_discovery/experimental/prompts/proposer_system_v2_0.txt, 2,077 bytes>",
  "messages": [
    {
      "role": "user",
      "content": "<examples/runs/quickstart_synthcount/packets/t01.json, 3,805 bytes>"
    }
  ]
}


The paper runs did not send the `v2.0` file. Their launch scripts send
`tools/fb1_launch_c/proposer_system_v2_9_FA-C.txt` through the development option
`--prompt-variant`; its first line declares its version tag, `v2.9.FA-C`, and the runs record
that tag and the file's sha256 (see the glossary entry "Prompt version and prompt variant").

In [8]:
paper_prompt = Path("tools/fb1_launch_c/proposer_system_v2_9_FA-C.txt")
paper_bytes = paper_prompt.read_bytes()
DOCUMENTED = "196f82b7a0f94e12a7cb8be2d07e776b1536aed11bf8ebd4bea1872c7002953c"  # USAGE.md
print(f"{paper_prompt}: {len(paper_bytes):,} bytes; first line: {paper_bytes.decode().splitlines()[0]!r}")
print(f"sha256 computed here: {hashlib.sha256(paper_bytes).hexdigest()}")
print(f"sha256 documented:    {DOCUMENTED}")

tools/fb1_launch_c/proposer_system_v2_9_FA-C.txt: 3,548 bytes; first line: '# prompt_version: v2.9.FA-C'
sha256 computed here: 196f82b7a0f94e12a7cb8be2d07e776b1536aed11bf8ebd4bea1872c7002953c
sha256 documented:    196f82b7a0f94e12a7cb8be2d07e776b1536aed11bf8ebd4bea1872c7002953c


## 5. Paragraphs appended for multi-candidate requests and archive reviews

In the paper protocol every round makes one request for three candidates, and every fifth round
is an **archive review round**, on which the language model may also ask for up to two models
that were **retired by evidence** to be refitted (**revive**). The prompt files ask for one
candidate and are never edited, so the runner appends paragraphs to the system text instead.
`count_proposer` defines them as constants:

- `COMPOSITION_PARAGRAPH_V2`, the **multi-candidate paragraph** (composition paragraph,
  version 2): reply with a JSON array of exactly `{limit}` candidates, three distinct,
  non-equivalent hypotheses, at least two differing in their primary functional form.
- `ARCHIVE_REVIEW_PARAGRAPH` (version `archive_review_paragraph_v1`): the reply becomes
  `{"candidates": [...], "revive": [...]}`.

Each constant's sha256 is the digest of its canonical bytes, with the `{limit}` placeholder
still in place:

In [9]:
COMPOSITION = count_proposer.COMPOSITION_PARAGRAPH_V2
REVIEW = count_proposer.ARCHIVE_REVIEW_PARAGRAPH
for name, text, recorded in (
        ("COMPOSITION_PARAGRAPH_V2", COMPOSITION, count_proposer.COMPOSITION_PARAGRAPH_V2_SHA256),
        ("ARCHIVE_REVIEW_PARAGRAPH", REVIEW, count_proposer.ARCHIVE_REVIEW_PARAGRAPH_SHA256)):
    computed = hashlib.sha256(text.encode()).hexdigest()
    print(f"{name} ({len(text.encode())} bytes):")
    print(textwrap.indent(textwrap.fill(text, 96), "    "))
    print(f"  sha256 computed here: {computed}")
    print(f"  sha256 constant:      {recorded}  equal: {computed == recorded}\n")

COMPOSITION_PARAGRAPH_V2 (525 bytes):
    Return a JSON array of exactly {limit} candidate objects, each with exactly the keys specified
    above and nothing else. Propose three distinct, non-equivalent hypotheses. At least two of the
    three candidates should differ meaningfully in their primary functional composition, rather than
    merely in parameterization or by adding or removing a correction term. At most one candidate may
    be a nested or incremental refinement of a high-posterior model, and only if such a refinement
    is specifically supported by the diagnostics.
  sha256 computed here: 782a30b74cd25adb6b51351096cf97e31a7375d8630d726ee8eb7d7949c35be6
  sha256 constant:      782a30b74cd25adb6b51351096cf97e31a7375d8630d726ee8eb7d7949c35be6  equal: True

ARCHIVE_REVIEW_PARAGRAPH (675 bytes):
    This is an archive review round. Besides your {limit} new candidates, you may ask for up to two
    archived models to be re-fit: choose only among archive entries whose outcome i

`composed_system_prompt` builds the text that is sent: the prompt file, then the composition
paragraph with `{limit}` replaced by 3, then, on a review round, the review paragraph. The
paragraph version is a module-level switch (`set_composition_paragraph`; `v1` is the default),
so the cell restores the previous setting afterwards. The identity block then also names the
paragraphs and the digest of the whole composed text (`composed_prompt_sha256`), which is what a
multi-candidate request record carries.

In [10]:
previous = count_proposer.set_composition_paragraph("v2")
try:
    three_text, three_identity = count_proposer.composed_system_prompt(3)
    review_text, review_identity = count_proposer.composed_system_prompt(3, review=True)
finally:
    count_proposer.set_composition_paragraph(previous)

base = prompt_bytes.decode()
print("k = 3:        file + composition paragraph:",
      three_text == base + COMPOSITION.replace("{limit}", "3"))
print("review round: file + composition + review paragraph:",
      review_text == three_text + REVIEW.replace("{limit}", "3"))
for label, block in (("identity block, k = 3", three_identity), ("identity block, review round", review_identity)):
    print(f"\n{label}:")
    for key, value in block.items():
        print(f"  {key:34} {value}")

k = 3:        file + composition paragraph: True
review round: file + composition + review paragraph: True

identity block, k = 3:
  prompt_version                     v2.0
  prompt_sha256                      f095edf1adb3b6533a9e68e73f98ca1dee3f23cd077e424b092393cb32b93aef
  proposal_limit                     3
  composed_prompt_sha256             83e36dca8703a0ab60a5570e4cfa9caaadac3c8f0766377ac1d3c3787b8b9317
  composition_paragraph_version      v2
  composition_paragraph_sha256       782a30b74cd25adb6b51351096cf97e31a7375d8630d726ee8eb7d7949c35be6

identity block, review round:
  prompt_version                     v2.0
  prompt_sha256                      f095edf1adb3b6533a9e68e73f98ca1dee3f23cd077e424b092393cb32b93aef
  proposal_limit                     3
  composed_prompt_sha256             c975831d6c3110a1a79af077a9f203210605b58fd0d7f17ceb0dfdd7db3ed91f
  composition_paragraph_version      v2
  composition_paragraph_sha256       782a30b74cd25adb6b51351096cf97e31a7375d8630d726ee

When each text is sent:

| Request | System text |
| --- | --- |
| one candidate per request (`--proposal-limit 1`, the default; this run) | the prompt file, unchanged |
| two or three candidates, paragraph `v1` (the default) | file + `COMPOSITION_PARAGRAPH` (v1) |
| three candidates with `--composition-paragraph v2` (every round of the paper protocol) | file + `COMPOSITION_PARAGRAPH_V2` |
| archive review round (`--archive-review-rounds`; rounds 5, 10, … in the paper) | file + `COMPOSITION_PARAGRAPH_V2` + `ARCHIVE_REVIEW_PARAGRAPH` |
| round-0 initial batch (`--initial-proposals K`), first wave / top-up waves | file + `BURST_PARAGRAPH` / `BURST_REFILL_PARAGRAPH` |

On this branch the ChemBench runner (`chembench_core`) has the `--composition-paragraph` and
`--archive-review-rounds` options; the synthetic count runner of notebook 01 does not, so with
`--proposal-limit 3` it appends the v1 paragraph. The ForceBench runner keeps its own copies of
the two paragraphs, with the same digests, in `tools/fb1_snapshot/fb1_proposer.py`, and the
BoxingGym runner has them at its paper tag. NeuronBench uses its own proposer
(`neuron_proposer.py`) and has no archive review rounds.

## 6. The bytes are part of the experiment

A prompt file, a paragraph constant and their digests identify an experiment the way a seed
does. Every run records `prompt_version` and `prompt_sha256` in `config.json` and in every
request record, and multi-candidate requests also record `composed_prompt_sha256` and the
paragraph digests. The ChemBench and NeuronBench launch scripts refuse to start when the prompt
file's digest differs from the one the paper used. A single changed byte gives a different
digest:

In [11]:
edited = prompt_bytes.replace(b"Propose one new candidate", b"Propose one new  candidate", 1)
print(f"the file as stored:         {hashlib.sha256(prompt_bytes).hexdigest()}")
print(f"with one extra space added: {hashlib.sha256(edited).hexdigest()}")

the file as stored:         f095edf1adb3b6533a9e68e73f98ca1dee3f23cd077e424b092393cb32b93aef
with one extra space added: 832c06721d0b42cf870230c5bd1e74ff64cbe7c9a463e7d6814bd9cb26563d29


So the prompt files in `src/bayes_model_discovery/experimental/prompts/`, the paper prompt in
`tools/fb1_launch_c/` and the paragraph constants in `count_proposer.py` are never edited in
place, not even to fix a typo: an edited file would silently change what an old version name
means, and runs that recorded the old digest could no longer be matched to their text. A new
text needs a new version; see
[`USAGE.md`](../USAGE.md#prompt-files-are-identifiers).

Next: [03 · expected information gain (EIG) decomposition](03_eig_decomposition.ipynb) or
[04 · Reading a run folder](04_reading_a_run_folder.ipynb).